# 04 — ML Models
**Saudi Tourism Intelligence Project**

3 Models:
1. Demand Forecasting (Prophet)
2. Tourist Segmentation (K-Means)
3. Spending Prediction (Gradient Boosting)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install prophet -q
import pandas as pd
import pickle
import os

base = '/content/drive/MyDrive/Saudi_Tourism_Project/data/clean'
models_path = '/content/drive/MyDrive/Saudi_Tourism_Project/models'
os.makedirs(models_path, exist_ok=True)

tourists = pd.read_csv(f'{base}/01_Tourists_CLEAN.csv')
kpi = pd.read_csv(f'{base}/03_KPI_CLEAN.csv')
overnight = pd.read_csv(f'{base}/02_Overnight_CLEAN.csv')
print('Setup done ✅')

## Model 1: Demand Forecasting (Prophet)

In [ ]:
from prophet import Prophet

monthly = tourists[tourists['Frequency']=='Monthly'].copy()
total_monthly = monthly.groupby('Period')['Tourists'].sum().reset_index()
total_monthly.columns = ['ds', 'y']
total_monthly['ds'] = pd.to_datetime(total_monthly['ds'])
total_monthly = total_monthly.sort_values('ds')

model_prophet = Prophet(yearly_seasonality=True,
                        seasonality_mode='multiplicative')
model_prophet.fit(total_monthly)

future = model_prophet.make_future_dataframe(periods=24, freq='MS')
forecast = model_prophet.predict(future)

with open(f'{models_path}/demand_forecast.pkl', 'wb') as f:
    pickle.dump(model_prophet, f)

forecast.to_csv(f'{base}/06_Demand_Forecast.csv', index=False)
print('Prophet Model saved ✅')
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(24)

## Model 2: Tourist Segmentation (K-Means)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

annual_t = tourists[tourists['Frequency']=='Annual'].copy()
annual_t['Period'] = annual_t['Period'].astype(int)
annual_o = overnight[overnight['Frequency']=='Annual'].copy()
annual_o['Period'] = annual_o['Period'].astype(int)
kpi['Year'] = kpi['Month'].str[:4].astype(int)

tourists_agg = annual_t.groupby(['Period','Type of tourist'])['Tourists'].sum().reset_index()
overnight_agg = annual_o.groupby(['Period','Type of tourist'])['Overnight Stays'].sum().reset_index()
kpi_agg = kpi.groupby(['Year','Type of tourist']).agg({
    'Average Length of Stay': 'mean',
    'Average Spending per Trip': 'mean',
    'Average Spending per Night': 'mean'
}).reset_index()
kpi_agg.columns = ['Period','Type of tourist','Avg_LOS','Avg_Spending_Trip','Avg_Spending_Night']

df = tourists_agg.merge(overnight_agg, on=['Period','Type of tourist'])
df = df.merge(kpi_agg, on=['Period','Type of tourist'])

features = ['Tourists','Overnight Stays','Avg_LOS','Avg_Spending_Trip','Avg_Spending_Night']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

with open(f'{models_path}/segmentation_kmeans.pkl', 'wb') as f:
    pickle.dump({'model': kmeans, 'scaler': scaler}, f)

df.to_csv(f'{base}/07_Tourist_Segments.csv', index=False)
print('K-Means Model saved ✅')

## Model 3: Spending Prediction (Gradient Boosting)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

kpi['Month_Num'] = kpi['Month'].str[-2:].astype(int)
le = LabelEncoder()
kpi['Tourist_Type_Enc'] = le.fit_transform(kpi['Type of tourist'])

features = ['Year','Month_Num','Tourist_Type_Enc',
            'Average Length of Stay','Average Spending per Night']
X = kpi[features]
y = kpi['Average Spending per Trip']

gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X, y)

y_pred = gb_model.predict(X)
print(f'R2: {r2_score(y, y_pred):.3f}')
print(f'MAE: {mean_absolute_error(y, y_pred):.2f} SAR')

with open(f'{models_path}/spending_gradient_boost.pkl', 'wb') as f:
    pickle.dump({'model': gb_model, 'encoder': le}, f)

print('Gradient Boosting Model saved ✅')